# Algorithms 09: Estimating Roots**Learning Objectives:**1. Estimate roots of functions using Bisection Search2. Estimate roots of functions using the Newton-Raphson Method3. Compare the convergence rates of both algorithms**Prerequisites:** Math 02 (Derivatives)**Estimated time:** 60 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapter 9

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "09_estimating_roots",    "level": 2,    "title": "Algorithms 09: Estimating Roots",    "prerequisites": [        "Math 02 (Derivatives)"    ],    "skills_taught": [        "lesson.algorithms_09_estimating_roots"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "10_single_variable_gd.ipynb"}SKILLS = [    {        "id": "lesson.algorithms_09_estimating_roots",        "title": "Lesson Algorithms 09 Estimating Roots",        "notebook": "09_estimating_roots.ipynb",        "level": 2    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "09_estimating_roots.ipynb",        "level": 2    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Math 02 (Derivatives).

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.algorithms_09_estimating_roots', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setupprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### Bisection Search> 📖 *From Algorithms Ch. 9:* To estimate the root using **bisection search**, we start with a lower bound and an upper bound, and repeatedly move one bound halfway to the other.If $f(x) = x^3 - 2$, we know $f(1) = -1$ and $f(2) = 6$. The root must be between 1 and 2.We test the midpoint $1.5$. $f(1.5) = 1.375 > 0$. Because it's positive, the root must be to its left. New bounds: $[1, 1.5]$.We repeat until the difference between bounds is less than a target `epsilon`.### Newton-Raphson Method> 📖 *From Algorithms Ch. 9:* We start with an initial guess and repeatedly update our guess to be the root of the *tangent line* to the function.The tangent line at $x_n$ has slope $f'(x_n)$. Its root is given by the update rule:$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$This method converges much faster, provided the derivative isn't zero.

### Comprehension Check ✅1. What is the fundamental requirement for the initial bounds $[a, b]$ in bisection search?2. What happens in Newton-Raphson if $f'(x_n) = 0$?<details><summary>💡 Explanation after a retrieval attempt</summary>1. $f(a)$ and $f(b)$ must have opposite signs, guaranteeing a root lies between them (Intermediate Value Theorem).2. The tangent line is horizontal and never hits the x-axis. The formula divides by zero and crashes.</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: A Single StepGiven $f(x) = x^2 - 5$. We want to find $\sqrt{5}$.1. Do one bisection step with bounds $[2, 3]$. What is the new midpoint and new bounds?2. Do one Newton-Raphson step with guess $x_0 = 2$. What is $x_1$?Compute these by hand, then verify with Python.

In [ ]:
def test_manual_steps():    # f(x) = x^2 - 5, f'(x) = 2x    midpoint = (2+3)/2 # 2.5    f_mid = midpoint**2 - 5 # 1.25 (positive, so replace upper bound)        x0 = 2    x1 = x0 - (x0**2 - 5) / (2*x0)        print("New bounds after 1 bisection:", [2, midpoint])    print("New guess after 1 Newton step:", x1)test_manual_steps()

---## Phase 1 — Algorithm Construction### Step 1: Bisection SearchImplement `bisection_search(f, lower, upper, epsilon=1e-5)`.

In [ ]:
def bisection_search(f, lower, upper, epsilon=1e-5):    '''Return the estimated root using bisection.'''    # Ensure f(lower) and f(upper) have opposite signs    # YOUR CODE HERE    pass# Test: cube root of 2# def f(x): return x**3 - 2# root = bisection_search(f, 1, 2)# print("Bisection root:", root)

### Step 2: Newton-Raphson MethodImplement `newton_raphson(f, f_prime, guess, epsilon=1e-5)`.

In [ ]:
def newton_raphson(f, f_prime, guess, epsilon=1e-5):    '''Return the estimated root using Newton-Raphson.'''    # Loop until abs(f(x)) < epsilon:    #   x = x - f(x)/f'(x)    # YOUR CODE HERE    pass# Test: cube root of 2# def f_prime(x): return 3*x**2# root = newton_raphson(f, f_prime, 2)# print("Newton root:", root)

---## Phase 2 — Experimental Verification### Comparing Convergence SpeedsLet's count how many iterations each method takes to achieve high precision (`1e-12`).

In [ ]:
def compare_convergence():    # Modify your functions above to return (root, num_iterations)    # and test them here for finding sqrt(2) with epsilon=1e-12    pass# compare_convergence()

### Pathological Case: Newton's NightmareNewton-Raphson is fast, but it can fail spectacularly. Consider $f(x) = x^{1/3}$.Its derivative is $\frac{1}{3}x^{-2/3}$. The update rule becomes $x_{n+1} = x_n - \frac{x_n^{1/3}}{\frac{1}{3}x_n^{-2/3}} = x_n - 3x_n = -2x_n$.If you start at $x_0 = 1$, the guesses are $1, -2, 4, -8, 16 \dots$ it spirals to infinity!Write a test showing this divergence.

In [ ]:
# YOUR CODE HERE to show Newton divergence on f(x) = x**(1/3)

---## Phase 3 — Olympiad Extension### Challenge: The Secant MethodWhat if you don't have the symbolic derivative $f'(x)$ for Newton's method?You can approximate the tangent using the two most recent guesses: $f'(x_n) \approx \frac{f(x_n) - f(x_{n-1})}{x_n - x_{n-1}}$.This is the **Secant Method**. Implement it!

In [ ]:
def secant_method(f, x0, x1, epsilon=1e-5):    '''Estimate root using secant method.'''    # YOUR CODE HERE    pass

### Chalkboard DefenseExplain the time complexity (convergence rate) of Bisection vs Newton. Bisection eliminates half the error each step. How does the error behave in Newton's method?<details><summary>💡 Sketch after a retrieval attempt</summary>Bisection has **linear convergence**. The error is halved each step: $E_{n+1} = 0.5 E_n$. It gains roughly 1 decimal point of precision every 3 steps.Newton has **quadratic convergence**. The error is squared each step: $E_{n+1} \propto E_n^2$. If the error is $0.1$, the next is $0.01$, then $0.0001$. The number of correct decimal places *doubles* with every step!</details>---📚 **Next:** Algorithms 10 (Single Variable Gradient Descent)

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.algorithms_09_estimating_roots', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='10_single_variable_gd.ipynb')